In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec  # Import for custom grid layout
from sklearn.model_selection import KFold
import pickle

from notebooks.imports import *
from config import dir_config
from src.utils import pmf_utils, glm_hmm_utils

In [ ]:
compiled_dir = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

In [ ]:
_TRIALS = "all_trials"
# _TRIALS = "all_trials_no_bias"
# _TRIALS = "all_trials_eq_prior"
# _TRIALS = "all_trials_uneq_prior"

data_path = f"global_glm_hmm_{_TRIALS}.pkl"
if "eq_prior" in _TRIALS:
	eq_only = True
else:
	eq_only = False

n_trial_back = 1

#### utils

In [ ]:
def get_train_test_split(n_trials, target_fold, k_folds):
	kf = KFold(n_splits=k_folds, shuffle=True, random_state=1)
	splits = [(train_idx, test_idx) for train_idx, test_idx in kf.split(np.arange(n_trials))]
	return splits[target_fold]

In [ ]:
def psychometric_fit(stimulus, choices, expected_choice_prob, ax, color, label, n_sample=10):
	data = {"signed_coherence": np.array(stimulus) * 100, "choice": choices}
	x_data, y_data, model, x_model, y_model = pmf_utils.get_psychometric_data(data)

	x_model_hat, y_model_hat = np.full((n_sample, len(x_model)), np.nan), np.full((n_sample, len(y_model)), np.nan)

	x_model_hat, y_model_hat = np.full((n_sample, len(x_model)), np.nan), np.full((n_sample, len(y_model)), np.nan)
	for idx_sample in range(n_sample):
		data_fitted = {"signed_coherence": np.array(stimulus) * 100, "choice": npr.binomial(1, expected_choice_prob)}
		x_data, y_data_hat, _, x_model_hat[idx_sample, :], y_model_hat[idx_sample, :] = pmf_utils.get_psychometric_data(data_fitted)

	ax.plot(x_data, y_data, "o", color=color)
	ax.plot(x_model, y_model, color=color, label=label)
	ax.plot(np.mean(x_model_hat, axis=0), np.mean(y_model_hat, axis=0), color=color, linestyle="--")
	ax.fill_between(np.mean(x_model_hat, axis=0), np.mean(y_model_hat, axis=0) - np.std(y_model_hat, axis=0), np.mean(y_model_hat, axis=0) + np.std(y_model_hat, axis=0), color=color, alpha=0.3)

	ax.set_xlim(min(x_data), max(x_data))
	ax.set_xlabel("Coherence")
	ax.set_ylabel("choices toRF")
	ax.set_title("Psychometric fits", fontsize=15)
	ax.legend()

In [ ]:
def plot_model_fits(model, choices, input, stimulus, mask, n_states, session_name, task_switch, trial_indices=None, eq_only=False):
	transition_matrix = model.transitions.params
	transition_matrix = np.exp(transition_matrix)[0]
	weights = -model.observations.params
	posterior_probs = model.expected_states(data=choices, input=input, mask=np.array(mask).reshape(-1, 1))[0]

	plt.figure(figsize=(14, 4))  # Wider figure for better spacing
	gs = gridspec.GridSpec(1, 4, width_ratios=[1, 1, 1, 2])  # Merge last two subplots

	cols = ["#ff7f00", "#4daf4a", "#377eb8"]  # Add more colors if needed for higher n_states
	plt.suptitle(session_name)
	# ----  First Subplot: Psychometric Curves ----
	ax1 = plt.subplot(gs[0])

	weighted_sum = np.sum(weights * input[None, :, :], axis=-1).T
	sigmoid_output = 1 / (1 + np.exp(-weighted_sum))
	glm_hmm_expected_choice_prob = np.sum(sigmoid_output * posterior_probs, axis=1, keepdims=True)

	if eq_only:
		equal_indices = np.where(mask)[0][:]
		if trial_indices is not None:
			equal_indices = np.intersect1d(np.concatenate(trial_indices), equal_indices)
		psychometric_fit(stimulus[equal_indices], choices[equal_indices], glm_hmm_expected_choice_prob[equal_indices], ax1, "#377eb8", label="Equal")
	else:
		equal_indices = np.where(mask)[0][:task_switch]
		unequal_indices = np.where(mask)[0][task_switch:]
		if trial_indices is not None:
			equal_indices = np.intersect1d(np.concatenate(trial_indices), equal_indices)
			unequal_indices = np.intersect1d(np.concatenate(trial_indices), unequal_indices)

		psychometric_fit(stimulus[equal_indices], choices[equal_indices], glm_hmm_expected_choice_prob[equal_indices], ax1, "#377eb8", label="Equal")
		psychometric_fit(stimulus[unequal_indices], choices[unequal_indices], glm_hmm_expected_choice_prob[unequal_indices], ax1, "#974810", label="Unequal")


	# ---- Second Subplot: GLM Weights ----
	# ax2 = plt.subplot(gs[1])
	# for k in range(n_states):
	# 	ax2.plot(np.arange(input.shape[1]), weights[k][0], marker="o", color=cols[k], linestyle="-", lw=1.5, label=f"State {k + 1}")

	# ax2.tick_params(axis="y", labelsize=10)
	# ax2.set_ylabel("GLM weight", fontsize=15)
	# ax2.set_xlabel("covariate", fontsize=15)
	# labels = ["Stimulus", "Bias"] \
	# 	+ [f"Prev Choice {t+1}" for t in range(n_trial_back)] \
	# 	+ [f"Prev Target {t+1}" for t in range(n_trial_back)]
	# ax2.set_xticklabels(labels, fontsize=12, rotation=15)

	# ax2.axhline(y=0, color="k", alpha=0.5, ls="--")
	# ax2.legend()
	# ax2.set_title("GLM weights", fontsize=15)

	# # ---- Third Subplot: Transition Matrix ----
	# ax3 = plt.subplot(gs[2])
	# ax3.imshow(transition_matrix, vmin=-0.8, vmax=1, cmap="bone")
	# for i in range(transition_matrix.shape[0]):
	# 	for j in range(transition_matrix.shape[1]):
	# 		ax3.text(j, i, str(np.around(transition_matrix[i, j], decimals=2)), ha="center", va="center", color="k", fontsize=12)

	# ax3.set_xlim(-0.5, n_states - 0.5)
	# ax3.set_ylim(n_states - 0.5, -0.5)
	# ax3.set_xticks(range(n_states))
	# ax3.set_yticks(range(n_states))
	# ax3.set_xlabel("state t+1", fontsize=15)
	# ax3.set_ylabel("state t", fontsize=15)
	# ax3.set_title("Generative transition matrix", fontsize=15)

	# ---- Fourth (Merged) Subplot: Posterior Probabilities ----
	ax4 = plt.subplot(gs[3])  # Merged across two columns
	for k in range(n_states):
		ax4.plot(posterior_probs[mask, k], label=f"State {k + 1}", lw=2, color=cols[k])

	ax4.set_ylim(-0.01, 1.01)
	ax4.set_yticks([0, 0.5, 1])
	ax4.tick_params(axis="y", labelsize=10)
	ax4.set_xlabel("trial #", fontsize=15)
	ax4.set_ylabel("p(state)", fontsize=15)
	ax4.axvline(x=task_switch, color="k", alpha=0.5, ls="--")
	ax4.legend()
	ax4.set_title("Posterior Probabilities", fontsize=15)

	plt.tight_layout()
	plt.show()

#### Load Data

In [ ]:
with open(Path(processed_dir, data_path), "rb") as f:
	glm_hmm = pickle.load(f)

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))

## Predictive Accuracy (TEST)

In [ ]:
max_n_state = 5
k_folds = 5

from math import isnan
from src.utils.glm_hmm_utils_cv import cross_validation_split

def predictive_accuracy(model, choices, input, mask):
    weights = -model.observations.params
    posterior_probs, weighted_sum = [], []
    for idx in range(len(choices)):
        posterior_probs.append(model.expected_states(data=choices[idx], input=input[idx], mask=np.array(mask[idx]).reshape(-1, 1))[0])
        weighted_sum.append(np.sum(weights * input[idx][None, :, :], axis=-1).T)
    sigmoid_output = 1 / (1 + np.exp(-np.concatenate(weighted_sum)))
    glm_hmm_expected_choice_prob = np.sum(sigmoid_output * np.concatenate(posterior_probs), axis=1, keepdims=True)
    
    log_diff, correct_prediction = [],[]
    for prob, choice in zip(glm_hmm_expected_choice_prob, np.concatenate(choices)):
        prob = np.clip(prob, 1e-6, 1 - 1e-6)  # Avoid log(0)
        if ~np.isnan(prob):
            # log_diff.append(np.log(prob) - np.log(0.5))
            if choice == 1:
                correct_prediction.append(prob>0.5)
                log_diff.append(np.log(prob) - np.log(0.5))
            elif choice == 0:
                correct_prediction.append(prob<0.5)
                log_diff.append(np.log(1 - prob) - np.log(0.5))
    return np.mean(correct_prediction), log_diff, correct_prediction


train_pred_accuracies = np.full((len(session_metadata), max_n_state, k_folds), np.nan)
test_pred_accuracies = np.full((len(session_metadata), max_n_state, k_folds), np.nan)
test_pred_log = {idx_session: {state:[] for state in range(1,max_n_state+1)} for idx_session in range(len(session_metadata))}
train_pred_log = {idx_session: {state:[] for state in range(1,max_n_state+1)} for idx_session in range(len(session_metadata))}
test_pred_corr = {idx_session: {state:[] for state in range(1,max_n_state+1)} for idx_session in range(len(session_metadata))}
train_pred_corr = {idx_session: {state:[] for state in range(1,max_n_state+1)} for idx_session in range(len(session_metadata))}
test_lls = np.full((len(session_metadata), max_n_state, k_folds), np.nan)
train_lls = np.full((len(session_metadata), max_n_state, k_folds), np.nan)
for state in range(1,max_n_state+1):
    for idx_session, session in enumerate(session_metadata["session_id"]):
        for fold in range(k_folds):
            model = glm_hmm["session_wise"]["models"][idx_session][state][fold]
            choices = glm_hmm["data"][session]["choices"].values.reshape(-1, 1)
            train_idx, test_idx = cross_validation_split(len(choices), idx_split=fold)
            input = np.array(
                glm_hmm["data"][session][
                    ["normalized_stimulus", "bias"] +
                    [f"prev_choice_{t+1}" for t in range(n_trial_back)] +
                    [f"prev_target_{t+1}" for t in range(n_trial_back)]
                ]
            )

            mask = glm_hmm["data"][session]["mask"]
            mask = np.ones_like(choices, dtype=bool) if mask is None else mask
            test_choices = [choices[test] for test in test_idx]
            test_input = [input[test] for test in test_idx]
            test_mask = [mask[test] for test in test_idx]
            test_pred_accuracies[idx_session, state-1, fold], log_diff_test, corr_pred_test = predictive_accuracy(model, test_choices, test_input, test_mask)
            test_pred_log[idx_session][state].append(log_diff_test)
            test_pred_corr[idx_session][state].append(corr_pred_test)

            train_choices = [choices[train] for train in train_idx]
            train_input = [input[train] for train in train_idx]
            train_mask = [mask[train] for train in train_idx]
            train_pred_accuracies[idx_session, state-1, fold], log_diff_train, corr_pred_train = predictive_accuracy(model, train_choices, train_input, train_mask)
            train_pred_log[idx_session][state].append(log_diff_train)
            train_pred_corr[idx_session][state].append(corr_pred_train)

            train_mask = [mask.values.reshape(-1,1) for mask in train_mask]
            train_lls[idx_session, state-1, fold] = model.log_likelihood(train_choices, inputs=train_input, masks=train_mask)
            
            test_mask = [mask.values.reshape(-1,1) for mask in test_mask]
            test_lls[idx_session, state-1, fold] = model.log_likelihood(test_choices, inputs=test_input, masks=test_mask)

train_pred_accuracies_foldmean = np.nanmean(train_pred_accuracies, axis=2)
train_pred_accuracies_mean = np.mean(train_pred_accuracies_foldmean, axis=0)
train_pred_accuracies_std = np.std(train_pred_accuracies_foldmean, axis=0)

test_pred_accuracies_foldmean = np.nanmean(test_pred_accuracies, axis=2)
test_pred_accuracies_mean = np.mean(test_pred_accuracies_foldmean, axis=0)
test_pred_accuracies_std = np.std(test_pred_accuracies_foldmean, axis=0)

test_ll_mean = np.mean(test_lls, axis=(0,2))
train_ll_mean = np.mean(train_lls, axis=(0,2))


In [ ]:
x = np.arange(max_n_state) + 1
fig, axs = plt.subplots(1, 1, figsize=(8, 6))

# Plot train accuracy
axs.plot(x, train_pred_accuracies_mean, marker='o', markersize=15, linewidth=3, color='k', label="Train set")
# axs.fill_between(x, train_pred_accuracies_mean - train_pred_accuracies_std,
#                     train_pred_accuracies_mean + train_pred_accuracies_std,
#                     color='k', alpha=0.2)

# Plot test accuracy
axs.plot(x, test_pred_accuracies_mean, marker='o', markersize=15, linewidth=3, color='b', label="Test set")
# axs.fill_between(x, test_pred_accuracies_mean - test_pred_accuracies_std,
#                     test_pred_accuracies_mean + test_pred_accuracies_std,
#                     color='b', alpha=0.2)

axs.set_xlabel("Number of States", fontsize=23)
axs.set_ylabel("Predictive Accuracy", fontsize=23)
axs.set_xticks(x)
axs.tick_params(axis='both', which='major', labelsize=15)
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)
axs.legend(fontsize=15)

plt.tight_layout()
plt.show()


In [ ]:
#
log_diff_means_state_2_correct, log_diff_means_state_2_incorr, log_diff_means_state_2 = [],[],[]
log_diff_means_state_5_correct, log_diff_means_state_5_incorr, log_diff_means_state_5 = [],[],[]
for idx_session in range(len(session_metadata)):
    log_diff_state_2 = np.concatenate(test_pred_log[idx_session][2])
    log_diff_state_5 = np.concatenate(test_pred_log[idx_session][5])
    test_pred_corr_state_2 = np.concatenate(test_pred_corr[idx_session][2])
    test_pred_corr_state_5 = np.concatenate(test_pred_corr[idx_session][5])
    # fig, axs = plt.subplots(1,2, figsize=(12,4), sharex=True, sharey=True)
    # axs[0].hist(log_diff_state_2[log_diff_state_2 < 0], bins=50, alpha=0.5, label='correct prediction', color='r')
    # axs[0].hist(log_diff_state_2[log_diff_state_2 > 0], bins=50, alpha=0.5, label='incorrect prediction', color='b')
    # axs[0].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} 2 states")
    # # axs[0].set_xlabel("log p - log 0.5")
    # axs[0].axvline(0, color='k', linestyle='--')
    # axs[0].legend()
    # # plt.xlabel("2 states (log p − log 0.5)", fontsize=15)
    # # plt.ylabel("5 states (log p − log 0.5)", fontsize=15)

    # axs[1].hist(log_diff_state_5[log_diff_state_5 < 0], bins=50, alpha=0.5, label='correct prediction', color='r')
    # axs[1].hist(log_diff_state_5[log_diff_state_5 > 0], bins=50, alpha=0.5, label='incorrect prediction', color='b')
    # axs[1].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} 5 states")
    # axs[1].set_xlabel("log p - log 0.5")    
    # axs[1].axvline(0, color='k', linestyle='--')
    # axs[1].legend()

    bin_edges_pos = np.arange(0, max(log_diff_state_2.max(), log_diff_state_5.max()) + 0.1, 0.1)
    bin_edges_neg = np.arange(min(log_diff_state_2.min(), log_diff_state_5.min()) - 0.1, 0, 0.1)

    fig, axs = plt.subplots(2, 1, figsize=(6, 8))

    # Top plot: correct predictions (log diff > 0)
    axs[0].hist(log_diff_state_2[log_diff_state_2>0], bins=bin_edges_pos, alpha=0.5, label='2 states', color='#FFBD86')
    axs[0].hist(log_diff_state_5[log_diff_state_2>0], bins=bin_edges_pos, alpha=0.5, label='5 states', color='#8FBAD9')
    axs[0].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} Correct prediction")
    axs[0].axvline(np.median(log_diff_state_2[log_diff_state_2>0]), color='#FFBD86', linestyle='--')
    axs[0].axvline(np.median(log_diff_state_5[log_diff_state_5>0]), color='#8FBAD9', linestyle='--')
    axs[0].legend()

    # Bottom plot: incorrect predictions (log diff < 0)
    axs[1].hist(log_diff_state_2[log_diff_state_2<0], bins=bin_edges_neg, alpha=0.5, label='2 states', color='#FFBD86')
    axs[1].hist(log_diff_state_5[log_diff_state_5<0], bins=bin_edges_neg, alpha=0.5, label='5 states', color='#8FBAD9')
    axs[1].axvline(np.median(log_diff_state_2[log_diff_state_2<0]), color='#FFBD86', linestyle='--')
    axs[1].axvline(np.median(log_diff_state_5[log_diff_state_5<0]), color='#8FBAD9', linestyle='--')
    axs[1].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} Incorrect prediction")
    # axs[1].axvline(0, color='k', linestyle='--')
    axs[1].legend()
    
#     log_diff_means_state_2_correct.append(np.mean(log_diff_state_2[log_diff_state_2 > 0]))
#     log_diff_means_state_2_incorr.append(np.mean(log_diff_state_2[log_diff_state_2 < 0]))
#     log_diff_means_state_2.append(np.mean(log_diff_state_2))
#     log_diff_means_state_5_correct.append(np.mean(log_diff_state_5[log_diff_state_5 > 0]))
#     log_diff_means_state_5_incorr.append(np.mean(log_diff_state_5[log_diff_state_5 < 0]))
#     log_diff_means_state_5.append(np.mean(log_diff_state_5))

    
# print(f"2 states: correct {np.mean(log_diff_means_state_2_correct)}, incorrect {np.mean(log_diff_means_state_2_incorr)}, all {np.mean(log_diff_means_state_2)}")
# print(f"5 states: correct {np.mean(log_diff_means_state_5_correct)}, incorrect {np.mean(log_diff_means_state_5_incorr)}, all {np.mean(log_diff_means_state_5)}")
# plt.figure()
# plt.scatter(log_diff_means_state_2_correct, log_diff_means_state_5_correct, color = 'r', label='Correct Predictions')
# plt.scatter(log_diff_means_state_2_incorr, log_diff_means_state_5_incorr, color = 'b', label='Incorrect Predictions')
# plt.scatter(log_diff_means_state_2, log_diff_means_state_5, color = 'k', label='All Predictions')
# plt.plot([-1.25,0.5],[-1.25,0.5],'k')
# plt.xlabel("2 states", fontsize = 15)
# plt.ylabel("5 states", fontsize = 15)
# plt.title("Mean Confidence in Observed Choices Above Chance\n (log p − log 0.5) Across sessions", fontsize=18)
# plt.legend()

In [ ]:
#
log_diff_means_state_2_correct, log_diff_means_state_2_incorr, log_diff_means_state_2 = [],[],[]
log_diff_means_state_5_correct, log_diff_means_state_5_incorr, log_diff_means_state_5 = [],[],[]
for idx_session in range(len(session_metadata)):
    log_diff_state_2 = np.concatenate(test_pred_log[idx_session][2])
    log_diff_state_5 = np.concatenate(test_pred_log[idx_session][5])
    test_pred_corr_state_2 = np.concatenate(test_pred_corr[idx_session][2])
    test_pred_corr_state_5 = np.concatenate(test_pred_corr[idx_session][5])

    fig, axs = plt.subplots(2, 1, figsize=(6, 8))

    # Top plot: correct predictions (test_pred_corr is True)
    axs[0].hist(log_diff_state_2[test_pred_corr_state_2], bins=30, alpha=0.5, label='2 states', color='#FFBD86')
    axs[0].hist(log_diff_state_5[test_pred_corr_state_5], bins=30, alpha=0.5, label='5 states', color='#8FBAD9')
    axs[0].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} Correct prediction")
    axs[0].axvline(np.median(log_diff_state_2[test_pred_corr_state_2]), color='#FFBD86', linestyle='--')
    axs[0].axvline(np.median(log_diff_state_5[test_pred_corr_state_5]), color='#8FBAD9', linestyle='--')
    axs[0].legend()

    # Bottom plot: incorrect predictions (test_pred_corr is False)
    axs[1].hist(log_diff_state_2[test_pred_corr_state_2], bins=30, alpha=0.5, label='2 states', color='#FFBD86')
    axs[1].hist(log_diff_state_5[test_pred_corr_state_2], bins=30, alpha=0.5, label='5 states', color='#8FBAD9')
    axs[1].axvline(np.median(log_diff_state_2[test_pred_corr_state_2]), color='#FFBD86', linestyle='--')
    axs[1].axvline(np.median(log_diff_state_5[test_pred_corr_state_2]), color='#8FBAD9', linestyle='--')
    axs[1].set_title(f"Session {session_metadata['session_id'][idx_session].replace('_', ' ')} Incorrect prediction")
    # axs[1].axvline(0, color='k', linestyle='--')
    axs[1].legend()
    break
    

In [ ]:
log_diff_state_2[test_pred_corr_state_2 > 0]

## Best state (Test LL)

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

# Assuming mean_ll_per_session is a NumPy array of shape (n_sessions, n_states_max)
test_ll = glm_hmm["session_wise"]["test_ll"]  # shape: [n_sessions, n_states, n_folds]
mean_ll_per_session = np.mean(test_ll, axis=-1)  # shape: [n_sessions, n_states]

n_sessions, max_n_state = mean_ll_per_session.shape

# Reshape into long-form DataFrame
df = pd.DataFrame({
    "n_states": np.repeat(np.arange(1, max_n_state + 1), n_sessions),
    "log_likelihood": mean_ll_per_session.T.flatten()
})

# Plot
sns.violinplot(x="n_states", y="log_likelihood", data=df)
plt.ylim(-100,-90)


In [ ]:
test_ll = np.mean(glm_hmm["session_wise"]["test_ll"], axis=(0, 2))
train_ll = np.mean(glm_hmm["session_wise"]["train_ll"], axis=(0, 2))
x = np.arange(max_n_state) + 1

fig, ax1 = plt.subplots()

# First y-axis: Train LL
ax1.plot(x, train_ll, marker='o', markersize=15, linewidth=3, color='b', label='Train LL')
ax1.set_xlabel("Number of States", fontsize=23)
ax1.set_ylabel("Train Log-likelihood", fontsize=23, color='b')
ax1.tick_params(axis='y', labelcolor='b', labelsize=15)
ax1.tick_params(axis='x', labelsize=15)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Second y-axis: Test LL
ax2 = ax1.twinx()
ax2.plot(x, test_ll, marker='o', markersize=15, linewidth=3, color=[0.6, 0.6, 0.6], label='Test LL')
ax2.set_ylabel("Test Log-likelihood", fontsize=23, color=[0.6, 0.6, 0.6])
ax2.tick_params(axis='y', labelcolor=[0.6, 0.6, 0.6], labelsize=15)
ax2.spines['top'].set_visible(False)

# Optional: Add legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
fig.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

plt.xticks(x)
plt.tight_layout()
plt.show()


#### Best fold for each session

In [ ]:
best_state_idx = 4
best_state = best_state_idx + 1

In [ ]:
best_fold_session_wise = []
for session in range(glm_hmm["session_wise"]["test_ll"].shape[0]):
	best_fold_session_wise.append(np.argmax(glm_hmm["session_wise"]["test_ll"][session, best_state_idx, :]))

#### Get train/test  split for each session

In [ ]:
from src.utils.glm_hmm_utils_cv import cross_validation_split
train_fold, test_fold = {}, {}

for idx_session, session in enumerate(session_metadata["session_id"]):
	choices = glm_hmm["data"][session]["choices"]
	best_fold = best_fold_session_wise[idx_session]

	train_indices, test_indices = cross_validation_split(len(choices), idx_split=best_fold)
	# Store the train and test indices for the current session
	train_fold[session] = train_indices
	test_fold[session] = test_indices

## Model fits and data recovery (train and test)

### Train Dataset

#### Psychometric Fits, GLM weights, transition matrix, p(state)

In [ ]:
for idx_session, session in enumerate(session_metadata["session_id"]):

	train_idx = train_fold[session]
	model = glm_hmm["session_wise"]["models"][idx_session][best_state][best_fold_session_wise[idx_session]]
	choices = glm_hmm["data"][session]["choices"].values.reshape(-1, 1)
	if "no_bias" in _TRIALS:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	else:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus", "bias"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	stimulus = glm_hmm["data"][session]["stimulus"]

	mask = glm_hmm["data"][session]["mask"]
	mask = np.ones_like(choices, dtype=bool) if mask is None else mask

	prob_toRF = glm_hmm["data"][session]["prob_toRF"]
	prob_toRF = prob_toRF[mask]
	indices = np.where((prob_toRF != 50) & ~np.isnan(prob_toRF))[0]
	task_switch = indices[0] if len(indices) > 0 else 0
	plot_model_fits(model, choices, input, stimulus, mask, best_state, session, task_switch, trial_indices=train_idx,eq_only=eq_only)

### Test Dataset

In [ ]:
for idx_session, session in enumerate(session_metadata["session_id"]):

	test_idx = test_fold[session]
	model = glm_hmm["session_wise"]["models"][idx_session][best_state][best_fold_session_wise[idx_session]]
	choices = glm_hmm["data"][session]["choices"].values.reshape(-1, 1)
	if "no_bias" in _TRIALS:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	else:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus", "bias"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	stimulus = glm_hmm["data"][session]["stimulus"]

	mask = glm_hmm["data"][session]["mask"]
	mask = np.ones_like(choices, dtype=bool) if mask is None else mask

	prob_toRF = glm_hmm["data"][session]["prob_toRF"]
	prob_toRF = prob_toRF[mask]
	indices = np.where((prob_toRF != 50) & ~np.isnan(prob_toRF))[0]
	task_switch = indices[0] if len(indices) > 0 else 0
	plot_model_fits(model, choices, input, stimulus, mask, best_state, session, task_switch, trial_indices=test_idx,eq_only=eq_only)

# Fit on whole session and save model

In [ ]:
# get parameter initializations for each session
init_params = {"glm_weights": {}, "transition_matrices": {}}
choices_session_wise, inputs_session_wise, masks_session_wise = [], [], []

# choose best state and best fold model parameters to initialize
for idx_session, session in enumerate(session_metadata["session_id"]):

	init_params["glm_weights"][idx_session] = glm_hmm["session_wise"]["models"][idx_session][best_state][best_fold_session_wise[idx_session]].observations.params
	init_params["transition_matrices"][idx_session] = glm_hmm["session_wise"]["models"][idx_session][best_state][best_fold_session_wise[idx_session]].transitions.params

	choices_session_wise.append(glm_hmm["data"][session]["choices"].values.reshape(-1, 1))
	if "no_bias" in _TRIALS:
		inputs_session_wise.append(np.array(
			glm_hmm["data"][session][
				["normalized_stimulus"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		))
	else:
		inputs_session_wise.append(np.array(
			glm_hmm["data"][session][
				["normalized_stimulus", "bias"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		))
	masks_session_wise.append(glm_hmm["data"][session]["mask"].values.reshape(-1, 1))

In [ ]:
models_session, fit_ll_session = glm_hmm_utils.session_wise_fit(choices_session_wise, inputs_session_wise, masks=masks_session_wise, n_sessions=len((session_metadata["session_id"])), init_params=init_params, n_states=best_state, n_iters=2500)

models_session_wise, fit_lls_session_wise = {}, {}
for idx_session, session_id in enumerate(session_metadata["session_id"]):
	models_session_wise[session_id] = models_session[idx_session]
	fit_lls_session_wise[session_id] = fit_ll_session[idx_session]

In [ ]:
for idx_session, session in enumerate(session_metadata["session_id"]):
	model = models_session_wise[session]
	choices = glm_hmm["data"][session]["choices"].values.reshape(-1, 1)
	if "no_bias" in _TRIALS:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	else:
		input = np.array(
			glm_hmm["data"][session][
				["normalized_stimulus", "bias"] +
				[f"prev_choice_{t+1}" for t in range(n_trial_back)] +
				[f"prev_target_{t+1}" for t in range(n_trial_back)]
			]
		)
	stimulus = glm_hmm["data"][session]["stimulus"]

	mask = glm_hmm["data"][session]["mask"]
	mask = np.ones_like(choices, dtype=bool) if mask is None else mask

	prob_toRF = glm_hmm["data"][session]["prob_toRF"]
	prob_toRF = prob_toRF[mask]
	indices = np.where((prob_toRF != 50) & ~np.isnan(prob_toRF))[0]
	task_switch = indices[0] if len(indices) > 0 else 0
	plot_model_fits(model, choices, input, stimulus, mask, best_state, session, task_switch, eq_only=eq_only)

In [ ]:
with open(Path(processed_dir, f"glm_hmm_{_TRIALS}_final.pkl"), "rb") as f:
	models_and_data = pickle.load(f)
models_session_wise = models_and_data["model"]["models"]
best_state = 2

In [ ]:
fig = plt.figure(figsize=(10, 4))  # Wider figure for better spacing
gs = gridspec.GridSpec(1, best_state)  # Two subplots with equal size

cols = ["#377eb8", "#974810"]  # Colors for awayRF and toRF
legend_added = {0: {cols[0]: False, cols[1]: False}, 1: {cols[0]: False, cols[1]: False}}  # Track legends per subplot

for idx_session, session_id in enumerate(session_metadata["session_id"]):
	model = models_session_wise[session_id]
	glm_weights = -np.array(model.observations.params).reshape(best_state, -1)
	prior_direction = 1 if glm_hmm["data"][session_id]["prob_toRF"].iloc[-1] > 50 else -1

	# if prior_direction == -1:
	# 	glm_weights = np.flip(glm_weights, axis=0)

	for k in range(best_state):
		ax = plt.subplot(gs[k])

		# Determine the color and label
		color = cols[0] if prior_direction == 1 else cols[1]
		label = "toRF prior session" if prior_direction == 1 else "awayRF prior session"

		# Only add legend once per color per subplot
		if not legend_added[k][color]:
			ax.plot(np.arange(2+2*n_trial_back), glm_weights[k], marker="o", color=color, linestyle="-", lw=1.5, label=label)
			legend_added[k][color] = True  # Mark color as used for this subplot
		else:
			ax.plot(np.arange(2+2*n_trial_back), glm_weights[k], marker="o", color=color, linestyle="-", lw=1.5)

		# Formatting
		ax.tick_params(axis="y", labelsize=10)
		ax.set_ylabel("GLM weight", fontsize=15)
		ax.set_xlabel("covariate", fontsize=15)
		ax.set_xticks(range(2+2*n_trial_back))
		ax.set_ylim([-3, 8])
		labels = ["Stimulus", "Bias"] \
			+ [f"Prev Choice {t+1}" for t in range(n_trial_back)] \
			+ [f"Prev Target {t+1}" for t in range(n_trial_back)]

		ax.set_xticklabels(labels, fontsize=12, rotation=15)

		ax.axhline(y=0, color="k", alpha=0.5, ls="--")

		ax.legend()  # Legend appears, but only one instance of each color per subplot
		ax.set_title(f"GLM weights of state {k + 1}", fontsize=15)

plt.show()

In [ ]:
best_state = 2

state_1 = []
state_2 = []

for idx_session, session_id in enumerate(session_metadata["session_id"]):
    model = models_session_wise[session_id]
    glm_weights = -np.array(model.observations.params).reshape(best_state, -1)
    state_1.append(glm_weights[0])
    state_2.append(glm_weights[1])
state_1 = np.array(state_1)
state_2 = np.array(state_2)

In [ ]:
plt.plot(np.mean(state_1, axis=0), label="Biased State", color="#E0590BC1", marker='o', markersize=12, linestyle='-', linewidth=4)
plt.plot(np.mean(state_2, axis=0), label="Unbiased State", color="#0FBD2CE0", marker='o', markersize=12, linestyle='-', linewidth=4)
plt.xticks(np.arange(2+2*n_trial_back), ["Stimulus", "Bias"] + [f"Prev. Choice" for t in range(n_trial_back)] + [f"Prev. Target" for t in range(n_trial_back)], rotation=15)
plt.tick_params(axis='both', which='major', labelsize=18)
plt.xlabel("Covariates", fontsize=23)
plt.ylabel("GLM Weights", fontsize=23)
# plt.title("GLM Weights for States 1 and 2", fontsize=20)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.legend(fontsize=18, bbox_to_anchor=(1.2, 1))

In [ ]:
fig = plt.figure(figsize=(10, 4))  # Wider figure for better spacing
gs = gridspec.GridSpec(1, best_state)  # Two subplots with equal size

cols = ["#377eb8", "#974810"]  # Colors for awayRF and toRF
legend_added = {0: {cols[0]: False, cols[1]: False}, 1: {cols[0]: False, cols[1]: False}}  # Track legends per subplot

for idx_session, session_id in enumerate(session_metadata["session_id"]):
	model = models_session_wise[session_id]
	glm_weights = -np.array(model.observations.params).reshape(best_state, -1)
	prior_direction = 1 if glm_hmm["data"][session_id]["prob_toRF"].iloc[-1] > 50 else -1

	# if prior_direction == -1:
	# 	glm_weights = np.flip(glm_weights, axis=0)

	for k in range(best_state):
		ax = plt.subplot(gs[k])

		# Determine the color and label
		color = cols[0] if prior_direction == 1 else cols[1]
		label = "toRF prior session" if prior_direction == 1 else "awayRF prior session"

		# Only add legend once per color per subplot
		if not legend_added[k][color]:
			ax.plot(np.arange(2+2*n_trial_back), glm_weights[k], marker="o", color=color, linestyle="-", lw=1.5, label=label)
			legend_added[k][color] = True  # Mark color as used for this subplot
		else:
			ax.plot(np.arange(2+2*n_trial_back), glm_weights[k], marker="o", color=color, linestyle="-", lw=1.5)

		# Formatting
		ax.tick_params(axis="y", labelsize=10)
		ax.set_ylabel("GLM weight", fontsize=15)
		ax.set_xlabel("covariate", fontsize=15)
		ax.set_xticks(range(2+2*n_trial_back))
		ax.set_ylim([-3, 8])
		labels = ["Stimulus", "Bias"] \
			+ [f"Prev Choice {t+1}" for t in range(n_trial_back)] \
			+ [f"Prev Target {t+1}" for t in range(n_trial_back)]

		ax.set_xticklabels(labels, fontsize=12, rotation=15)

		ax.axhline(y=0, color="k", alpha=0.5, ls="--")

		ax.legend()  # Legend appears, but only one instance of each color per subplot
		ax.set_title(f"GLM weights of state {k + 1}", fontsize=15)

plt.show()

In [ ]:
# # store data and models for session-wise
# session_wise_fits = {
# 	"models": models_session_wise,
# 	"fit_lls": fit_lls_session_wise,
# }

# models_and_data = {
# 	"model": session_wise_fits,
# 	"data": glm_hmm["data"],
# }

# with open(Path(processed_dir, f"glm_hmm_{_TRIALS}_final.pkl"), "wb") as f:
# 	pickle.dump(models_and_data, f)